In [1]:
import pandas as pd
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import importlib
import preprocessing as p
importlib.reload(p)

from sklearn.model_selection import GridSearchCV
from preprocessing import ordinal_scales
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, PowerTransformer
from sklearn.model_selection import cross_val_score

In [2]:
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

X=df_train.drop(columns=['SalePrice'])
y=df_train['SalePrice']
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.2,random_state=42)

test_ids = df_test['Id'].copy()

In [3]:
modeles = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=10.0),
    'Lasso':            Lasso(alpha=0.001),
    'ElasticNet':       ElasticNet(alpha=0.001, l1_ratio=0.5),
    'RandomForest':     RandomForestRegressor(n_estimators=300, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'KNN':              KNeighborsRegressor(n_neighbors=10),
}

In [4]:
def features(df):
    df=df.copy()
    df['TotalSF']=df['TotalBsmtSF']+df['GrLivArea']
    df['RatioArea']=df['TotalBsmtSF']/df['LotArea']
    df['RoomSize']=df['GrLivArea']/df['TotRmsAbvGrd']
    df['BedroomSize']=df['BedroomAbvGr']/df['GrLivArea']
    df['HouseAge']=df['YrSold']-df['YearBuilt']
    df['HouseRemod']=df['YrSold']-df['YearRemodAdd']
    return df

from sklearn.preprocessing import FunctionTransformer

feature_eng = FunctionTransformer(features)

In [5]:
X=features(X)

In [6]:
ordinal_cols = list(ordinal_scales.keys())

num_cols = X.select_dtypes(include='number').columns.tolist()

# toutes les colonnes texte/bool, MOINS les ordinales
cat_cols = X.select_dtypes(include=['bool', 'object']).columns.tolist()
nominal_cols = [c for c in cat_cols if c not in ordinal_cols]

num_cols = [c for c in num_cols if c != 'Id']

In [7]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson')),
])

ordinal_pipe= Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='None')),
('OrdinalEncoder',OrdinalEncoder(categories=[ordinal_scales[c] for c in ordinal_cols]))])

nominal_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='None')),('OneHotEncoder',OneHotEncoder(handle_unknown='ignore', sparse_output=False))])


In [8]:
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('ord', ordinal_pipe, ordinal_cols),
    ('nom', nominal_pipe, nominal_cols),
])

In [9]:
lasso_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('model', Lasso(max_iter=10000)),   # max_iter élevé pour éviter le warning de convergence
])

param_grid_lasso = {'model__alpha': [0.0001, 0.001, 0.005, 0.01, 0.05]}


grid = GridSearchCV(lasso_pipe, param_grid_lasso, cv=5,
                    scoring='neg_mean_squared_error')
grid.fit(X, np.log1p(y))

print("meilleur alpha :", grid.best_params_)
print("RMSE           :", np.sqrt(-grid.best_score_))

meilleur alpha : {'model__alpha': 0.001}
RMSE           : 0.1251821713113441


In [10]:
Boost_pipe=Pipeline([('preprocessing',preprocessor),('GradientBoosting',GradientBoostingRegressor(random_state=42))])
Boost_pipe.fit(X, np.log1p(y)) 

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('GradientBoosting', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](86,)","['Id','MSSubClass','MSZoning',...,'BedroomSize','HouseAge','HouseRemod']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,86
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This

In [11]:
df_test=features(df_test)

In [12]:
predictions_log = grid.predict(df_test)
predictions = np.expm1(predictions_log)   # retour en dollars
soumission = pd.DataFrame({ 'Id': test_ids,
'SalePrice': predictions
})
soumission.to_csv('soumission_withFE_Lasso.csv', index=False)

In [15]:
from sklearn.ensemble import VotingRegressor

VotingRegressor([('lasso', grid.best_estimator_), ('boost', Boost_pipe)],
                weights=[3, 1])   # Lasso compte 3×, Boost 1×

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('lasso', ...), ('boost', ...)]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.","[3, 1]"
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time

In [16]:
ensemble.fit(X,np.log1p(y))
predictions_log = ensemble.predict(df_test)
predictions = np.expm1(predictions_log)   # retour en dollars
soumission = pd.DataFrame({ 'Id': test_ids,
'SalePrice': predictions
})
soumission.to_csv('soumission_withFE_Ensemble.csv', index=False)

In [23]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=3,
                   subsample=0.8, random_state=42)

xgb_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('model', xgb),
])

xgb_pipe.fit(X,np.log1p(y))

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](86,)","['Id','MSSubClass','MSZoning',...,'BedroomSize','HouseAge','HouseRemod']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,86
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of 

In [24]:
scores = cross_val_score(xgb_pipe, X, np.log1p(y), cv=5, scoring='neg_mean_squared_error')
print("RMSE ensemble CV :", np.sqrt(-scores.mean()))

RMSE ensemble CV : 0.12319522892947023


In [25]:
predictions_log = xgb_pipe.predict(df_test)
predictions = np.expm1(predictions_log)   # retour en dollars
soumission = pd.DataFrame({ 'Id': test_ids,
'SalePrice': predictions
})
soumission.to_csv('soumission_with_XGB.csv', index=False)

In [26]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'model__n_estimators': randint(500, 2000),
    'model__learning_rate': uniform(0.01, 0.1),
    'model__max_depth': randint(3, 6),
    'model__subsample': uniform(0.7, 0.3),          # entre 0.7 et 1.0
    'model__colsample_bytree': uniform(0.7, 0.3),
    'model__min_child_weight': randint(1, 6),
}

search = RandomizedSearchCV(xgb_pipe, param_dist, n_iter=50, cv=5,
                            scoring='neg_mean_squared_error', random_state=42, n_jobs=-1)
search.fit(X, np.log1p(y))
print(search.best_params_)
print("RMSE :", np.sqrt(-search.best_score_))

{'model__colsample_bytree': np.float64(0.7190675050858071), 'model__learning_rate': np.float64(0.041098232171566225), 'model__max_depth': 3, 'model__min_child_weight': 3, 'model__n_estimators': 1183, 'model__subsample': np.float64(0.8912672414065639)}
RMSE : 0.12014558500909832


In [27]:
predictions_log = search.predict(df_test)
predictions = np.expm1(predictions_log)   # retour en dollars
soumission = pd.DataFrame({ 'Id': test_ids,
'SalePrice': predictions
})
soumission.to_csv('soumission_with_XGB_tuned.csv', index=False)

In [ ]:
for w in [[1,4],[1,3],[1,2],[1,1], [2,1], [3,1], [4,1], [5,1]]:
    ens = VotingRegressor([('lasso', grid.best_estimator_), ('xgb', search.best_estimator_)], weights=w)
    scores = cross_val_score(ens, X, np.log1p(y), cv=5, scoring='neg_mean_squared_error')
    print(f"poids {w} : {np.sqrt(-scores.mean()):.4f}")

In [ ]:
VotingRegressor([('lasso', grid.best_estimator_), ('boost', Boost_pipe)],
                weights=[3, 1])   # Lasso compte 3×, Boost 1×